In [22]:
# !pip uninstall deepspeed --quiet
# !pip install deepspeed --quiet

In [23]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [1]:
%cd /content/drive/MyDrive/CAISI_phase2/SaLoRA-finetuning/SaLoRA-Repo

/content/drive/MyDrive/CAISI_phase2/SaLoRA-finetuning/SaLoRA-Repo


In [25]:
# !pip install --no-cache-dir torch==2.8.0+cu126 torchvision==0.15.1+cu126 torchaudio==2.8.0+cu126 --index-url https://download.pytorch.org/whl/cu126

In [26]:
!pip install -r requirements.txt

KeyboardInterrupt: 

In [ ]:
!pip install 'flash_attn<2.8' --quiet

In [ ]:
import numpy, transformers, vllm, trl, flash_attn

In [ ]:
print("numpy:", numpy.__version__)
print(transformers.__version__)
print("vllm:", vllm.__version__)
print("trl:", trl.__version__)

In [ ]:
#import lora_train_act

# lora_train_act.main({
#   "name": "my_run",
#   "rank": 16,
#   "rs": 32,
#   "ds": 32,
#   "bs": 16,
#   "port": 1234,
#   "rank_s": 32,
#   "lr": 1e-4,
#   "wd": 0.0,
#   "more_module": False,
# })

In [ ]:
# %tb args = lora_train_act.parse_args()

In [2]:
from safetensors.torch import load_file, save_file
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
import os
from lowrank_prune.lib.model_wrapper_low import *
from lowrank_prune.lib.data import get_loaders
from peft import LoraConfig, get_peft_model
import time
import sys
sys.path.append(".")
from trl import SFTTrainer                          #specialized trainer for supervised fine-tuning of LMs
from datasets import load_dataset, Dataset          #load and manipulate datasets - from HuggingFace
from transformers import Trainer, TrainingArguments #training interface - from HuggingFace
from fastchat.conversation import get_conv_template #formatting conversations with LMs
from tqdm import tqdm                               #"Taqadum" - display progress bars for loops


In [3]:
import argparse
def parse_args():
    parser = argparse.ArgumentParser(description='Brisque') #built-in Python library to handle CLI arguments
    parser.add_argument('--name', default="output_alphca_chat7b_act12", help='path to input image file')      #output folder name
    parser.add_argument('--rank', type=int, default=16, help='path to input image file')                      #LoRA rank
    parser.add_argument('--rs', type=int, default=32, help='path to input image file')                        #rank for safety subspace
    parser.add_argument('--ds', type=int, default=32, help='path to input image file')                        #rank for task dataset
    parser.add_argument('--bs', type=int, default=16, help='path to input image file')                        #batch size for training
    parser.add_argument('--port', type=int, default=1234, help='path to input image file')                    #master port for distributed training (torch)
    parser.add_argument('--rank_s', type=int, default=32, help='path to input image file')                    #secondary rank param for LoRA
    parser.add_argument('--lr', type=float, default=1e-4, help='path to input image file')                    #learning rate
    parser.add_argument('--wd', type=float, default=0.0, help='path to input image file')                     #weight decay
    parser.add_argument('--more_module', action='store_true', default=False, help='path to input image file') #flag for whether to include additional LoRA modules
    args = parser.parse_known_args()[0] #parses arguments; each becomes an attribute of args (learning rate = args.lr)
    return vars(args)

In [4]:
args = parse_args()

In [ ]:
#for k, v in vars(args).items():
#    print(k, v)

In [5]:
aligned_model_path = "meta-llama/Meta-Llama-3.1-8B-Instruct"
#Single GPU set-up
os.environ['MASTER_ADDR'] = 'localhost'
os.environ['WORLD_SIZE'] = '1'

In [6]:
#Data preprocessing - function splits batches of examples from dataset into fixed-size blocks of tokens
def group_texts(examples, block_size=128):
    # Concatenate all texts.
    # examples[k] = list of lists, each list contains the tokens of one sample
    concatenated_examples = {k: sum(examples[k], []) for k in examples.keys()}
    total_length = len(concatenated_examples[list(examples.keys())[0]]) #get total number of tokens based on first key ('input_ids')
    total_length = (total_length // block_size) * block_size # length is divisible by block size
    # Split by chunks of max_len.
    result = {
        k: [t[i : i + block_size] for i in range(0, total_length, block_size)] # builds list of block-sized chunks
        for k, t in concatenated_examples.items()
    }
    result["labels"] = result["input_ids"].copy() # Next-token prediction - use the correct previous tokens as input
    return result
# Returns a dict like:
#{
# 'input_ids': [[123, 324, ..., 128 (tokens)], [...], ...],
# 'attention_mask': [[1, 1, ...], [...], ...],
# 'labels': [[103, 231, ..., 128 tokens], [...], ...]
#}

In [7]:
#def main(args_dict=None):
#    if args_dict is None:
args_dict = parse_args()  # This returns a dict
args = argparse.Namespace(**args_dict)  # Convert to namespace
 #   else:
 #     class Args: pass
#      args = Args()
    #  for k, v in args_dict.items():
   # setattr(args, k, v)
os.environ['MASTER_PORT'] = str(args.port)
#Creates directory path for outputs
args.name = 'out/'+str(args.rs)+'_'+str(args.ds)+str(args.more_module)+'llama3.1_alpaca_output/llama3.1_' + args.name +'_rank'+str(args.rank_s)
try:
    #Creates folder if it does not yet exist
    os.mkdir('out/'+str(args.rs)+'_'+str(args.ds)+str(args.more_module)+'llama3.1_alpaca_output/')
    print("Make dir "+'out/'+str(args.rs)+'_'+str(args.ds)+str(args.more_module)+'llama3.1_alpaca_output/')
except:
    print("Already Exists")


Already Exists


In [8]:
#Load pre-trained tokenizer from aligned_model_path
tokenizer = AutoTokenizer.from_pretrained(aligned_model_path)
tokenizer.pad_token_id = tokenizer.eos_token_id #LlaMA models don't have a dedicated <pad> token
model = AutoModelForCausalLM.from_pretrained(aligned_model_path, device_map="cuda:0") #put model on GPU 0
if args.more_module: #choose to adapt more layers (key, feed-forward up/down projections)
    target_module_list = ["q_proj", "k_proj", "v_proj", "up_proj", "down_proj"]
else:
    target_module_list = ["q_proj","v_proj"]
divide_num = len(target_module_list) #store number of target modules

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

In [9]:
# print(type(args[0]))
def tokenize_function(examples):
    return tokenizer(examples["text"])
adapter_name ='default'
rank = args.rank
n_iter=30
peft_config = LoraConfig(
    r = rank,
    lora_alpha = rank, # lora_alpha should match r to maintain scaling = 1
    lora_dropout = 0,
    inference_mode=False,
    init_lora_weights=False, # PiSSA initialization
    target_modules = target_module_list,
    task_type="CAUSAL_LM",
)

t1 = time.time()
model = make_Act(model, verbose=False) #wraps model so linear layers can record activations
model.requires_grad_(False)            #freeze base model weights
model.seqlen = 4096                    #set max sequence length
clear_act_buffer(model)                #wipe any old stored activations

#initially, ActLinear layers do not record activates; activated one layer at a time to save memory
for name, module in model.named_modules():
    if isinstance(module, ActLinear):
        module.record_activation = False
        module.clear_act_buffer()

#Load data for collecting activations
dataloader, _ = get_loaders( #regular instruction dataset
    "alpaca_cleaned_no_safety",
    nsamples=128,
    seed=0,
    seqlen=model.seqlen,
    tokenizer=tokenizer,
    disentangle=True,
)
dataloader_safe, _ = get_loaders(
    'align_short',
    nsamples=128,
    seed=0,
    seqlen=model.seqlen,
    tokenizer=tokenizer,
    disentangle=True,
)

num_hidden_layers = model.config.num_hidden_layers
weight_list = {}
current_num = -1

In [ ]:
for layer in range(num_hidden_layers):
        layer_filter_fn = (
            lambda x: f"layers.{layer}." in x
        )  ### TODO # hack for llama series

    # enable recording for the current layer.
        for name, module in model.named_modules():
            if layer_filter_fn(name) and isinstance(module, ActLinear):
                # print("enabling recording for ", name)
                # if 'v_proj' in name or 'q_proj' in name:
                module.record_activation = True
                module.clear_act_buffer()

        #Run data through model and collect activations
        for batch in dataloader_safe:
            inp, tar = batch[0].to("cuda:0"), batch[1].to("cuda:0")
            assert True, "should run in disentangle mode"
            mask = tar.ne(-100)           #ignore padded tokens
            with set_mask(model, mask):   #ensures only valid tokens are used
                model(inp)                #triggers forward passes --> activations get recorded

        #For each module in current layer: match its name in the target_module_list and concatenate all collected activations into one big tensor
        for name, module in model.named_modules():
            if layer_filter_fn(name) and isinstance(module, ActLinear):
                print("Module name:", name)
                # print(module.activation_norms)
                f_name = ""
                for t_name  in target_module_list:
                    if t_name in name:
                        f_name = t_name
                        break
                if len(f_name)==0:
                    print("Wrong name ", name)
                    continue
                current_num += 1
                layer_n =f_name
                module.activation_norms = torch.cat(module.activation_norms, dim=0).to(
                    "cuda:0"
                )  # size * d_in
                score = ( #multiplication producing a matrix representing how activations align with weight directions
                    module.activation_norms @ module.base.weight.data.T
                )  # (size * d_in) @ (d_out * d_in).T --> (size, d_out)
                d_out, d_in = module.base.weight.data.shape
                total_rank = min(d_out, d_in)
                print(
                    f"remaining: rank {name} = {total_rank - rank} / {total_rank}"
                )
                # for removing from the top
                U, S, V = torch.svd_lowrank(
                    score.float(), q=args.rs, niter=n_iter
                )  # (size, r) (r) (d_out, r)
                V =  V.type(module.base.weight.data.dtype) #save V as lora matrix initializer
                #save each V matrix inot a dictionary weight_list, keyed by layer name + index
                #These will be injected into LoRA modules instead of random initialization
                weight_list[layer_n+'_'+str(current_num//divide_num)+'_V'] = V

        #After finishing a layer, stop recording and clear buffers before moving to next layer
        for name, module in model.named_modules():
            if layer_filter_fn(name) and isinstance(module, ActLinear):
                # print("disabling recording for ", name)
                module.record_activation = False
                module.clear_act_buffer()


In [ ]:
current_num = -1
for layer in range(num_hidden_layers):
    layer_filter_fn = (
        lambda x: f"layers.{layer}." in x
    )  ### TODO # hack for llama series

# enable recording for the current layer.
    for name, module in model.named_modules():
        if layer_filter_fn(name) and isinstance(module, ActLinear):
            # print("enabling recording for ", name)
            # if 'v_proj' in name or 'q_proj' in name:
            module.record_activation = True

    # forward pass and get activation records.

# with torch.no_grad():
  #runs model forward on data, activations from ActLinear models are stored
    for batch in dataloader:
        inp, tar = batch[0].to("cuda:0"), batch[1].to("cuda:0")
        assert True, "should run in disentangle mode"
        mask = tar.ne(-100)
        with set_mask(model, mask):
            model(inp)

    for name, module in model.named_modules():
        if layer_filter_fn(name) and isinstance(module, ActLinear):
            # print("Module name:", name)
            # print(module.activation_norms)
            f_name = ""
            for t_name  in target_module_list:
                if t_name in name:
                    f_name = t_name
                    break
            if len(f_name)==0:
                # print("Wrong name ", name)
                continue
            current_num += 1
            layer_n = f_name
            module.activation_norms = torch.cat(module.activation_norms, dim=0).to(
                "cuda:0"
            )  # size * d_in
            score = (
                module.activation_norms @ module.base.weight.data.T
            )  # (size * d_in) @ (d_out * d_in).T --> (size, d_out)
            d_out, d_in = module.base.weight.data.shape
            total_rank = min(d_out, d_in)
            print(
                f"remaining: rank {name} = {total_rank - rank} / {total_rank}"
            )

            # for removing from the bottom
            U, S, V = torch.svd_lowrank(
                score.float(), q=args.ds, niter=n_iter
            )  # (size, r) (r) (d_out, r)
            V =  V.type(module.base.weight.data.dtype)
            U2, S2, V2 = torch.svd_lowrank(
                module.base.weight.data.float(), q=rank, niter=n_iter
            )  # n(size, r) (r) (d_out, r)
            U2 =  U2.type(module.base.weight.data.dtype)
            S2 =  S2.type(module.base.weight.data.dtype)
            V2 =  V2.type(module.base.weight.data.dtype)
            safeV = weight_list[layer_n+'_'+str(current_num//divide_num)+'_V']
            weight_list[layer_n+'_'+str(current_num//divide_num)+'lora_C'] = torch.eye(safeV.size(0),dtype=module.base.weight.dtype, device=module.base.weight.device) - weight_list[layer_n+'_'+str(current_num//divide_num)+'_V'] @ weight_list[layer_n+'_'+str(current_num//divide_num)+'_V'].T

            #construct LoRA A and B matrices from SVD factors
            #low-rank adapters that approximate effect of full weight update
            weight_list[layer_n+'_'+str(current_num//divide_num)+'lora_B'] = U2 @ torch.diag(torch.sqrt(S2))
            weight_list[layer_n+'_'+str(current_num//divide_num)+'lora_B'] = V @ V.T @ weight_list[layer_n+'_'+str(current_num//divide_num)+'lora_B']
            weight_list[layer_n+'_'+str(current_num//divide_num)+'lora_A'] = torch.diag(torch.sqrt(S2)) @ V2.T
            module.base.weight.data.sub_(
                weight_list[layer_n+'_'+str(current_num//divide_num)+'lora_C'] @ weight_list[layer_n+'_'+str(current_num//divide_num)+'lora_B'] @ weight_list[layer_n+'_'+str(current_num//divide_num)+'lora_A'])
            weight_list[layer_n+'_'+str(current_num//divide_num)+'weight'] = module.base.weight.data
            # print(weight_list[layer_n+'_'+str(current_num//divide_num)+'lora_B'].size(), weight_list[layer_n+'_'+str(current_num//divide_num)+'lora_A'].size())
        # torch.cuda.empty_cache()

# disable recording for the current layer.
    for name, module in model.named_modules():
        if layer_filter_fn(name) and isinstance(module, ActLinear):
            # print("disabling recording for ", name)
            module.record_activation = False
            module.clear_act_buffer()

In [ ]:
t2 = time.time()
print("Time cost", t2-t1)
model = revert_Act_to_Linear(model)         #restores model from ActLinear to standard nn.Linear
model.zero_grad()
model = get_peft_model(model, peft_config)  #wraps model with LoRA hooks so only low-rank adapters are trainable
model.print_trainable_parameters()          #sanity check; confirm only LoRA params are active
print(model.parameters())

In [ ]:
 #replace random initialization of A and B matrices with the computed ones that are safe aligned
#also attaches lora_C
#current_num = 0
print(weight_list.keys())

module_counters = {t_name: 0 for t_name in target_module_list}

for name, module in model.named_modules():
    for t_name in target_module_list:
        if t_name in name:
            layer_idx = module_counters[t_name] // divide_num

            if 'lora_A.default' in name:
                key = f'{t_name}_{layer_idx}lora_A'
                if key in weight_list:
                    module.weight.data = weight_list[key]
                    print(f"Set {name} with {key}")
                else:
                    print(f"Warning: {key} not found")

            elif 'lora_B.default' in name:
                key = f'{t_name}_{layer_idx}lora_B'
                if key in weight_list:
                    module.weight.data = weight_list[key]
                    print(f"Set {name} with {key}")
                else:
                    print(f"Warning: {key} not found")

            elif t_name + '.' not in name:  # Main module (not lora_A or lora_B)
                key = f'{t_name}_{layer_idx}lora_C'
                if key in weight_list:
                    module.lora_C = weight_list[key].clone()
                    weight_list[key] = module.lora_C.clone()
                    module.requires_grad = False
                    print(f"Set {name} lora_C with {key}")
                else:
                    print(f"Warning: {key} not found")

                module_counters[t_name] += 1

            break

#for name, module in model.named_modules():
#   for t_name  in target_module_list:
#        if t_name in name:
 #           if 'lora_A.default' in name:
  #              # print(name)
   #             module.weight.data = weight_list[t_name+'_'+str(current_num//(2*divide_num))+'lora_A']
    #            current_num +=1
     #       if 'lora_B.default' in name:
      #          module.weight.data = weight_list[t_name+'_'+str(current_num//(2*divide_num))+'lora_B']
       #         current_num +=1
#
 #           if t_name+'.' not in name:
  #              # print(name)
   #             module.lora_C = weight_list[t_name+'_'+str(current_num//4)+'lora_C'].clone()
    #            weight_list[t_name+'_'+str(current_num//4)+'lora_C'] = module.lora_C.clone()
     #           module.requires_grad = False #ensures C stays frozen during fine-tuning
      #      break

In [ ]:
weight_list['divide_num'] = divide_num

#saves A, B, and C matrices so they can be reloaded later
save_path = args.name+'_'+str(rank) + '_' + str(args.lr) + '_' + str(args.wd) + 'lora_ABC.pt'
os.makedirs(os.path.dirname(save_path), exist_ok = True)
torch.save(weight_list, save_path)
print(f"Saved to {save_path}")

In [ ]:

del weight_list
torch.cuda.empty_cache()
prompts = []

In [13]:
import logging
import datasets
import csv
logging.info("loading dataset")

dataset = datasets.load_from_disk("/content/drive/MyDrive/CAISI_phase2/qlora-finetuning/tune_datasets/alpaca_clean_sample_5k/")
train_ds = dataset['train']
val_ds = dataset['test']

rows = []
for i in range(len(train_ds)):
  sample = train_ds[i]
  input = sample['instruction'] + ': ' + sample['input']
  output = sample['output']
  rows.append([i, input, output])

header = ['index', 'prompt', 'response']
with open("/content/drive/MyDrive/CAISI_phase2/SaLoRA-finetuning/SaLoRA-Repo/lowrank_prune/data/alpaca_5k_train.csv", "w") as f:
  writer = csv.writer(f)
  writer.writerow(header)
  for row in rows:
    writer.writerow(row)

In [ ]:
#dataset = load_dataset
#count = 0
#for line in tqdm(dataset): #reformat into instruction input / output style
 #   count += 1
  #  prompts.append({'text':f"[INST]{line['instruction'].strip()} {line['input']}[/INST]{line['output']}"})


In [ ]:
df = Dataset.from_list(prompts)
training_args = TrainingArguments(
    args.name+"_"+str(rank)+"_"+str(args.lr)+'_'+str(args.wd)+"/",
    evaluation_strategy = "no",
    learning_rate=args.lr,
    weight_decay=args.wd,
    save_steps = 500,

    per_device_train_batch_size=args.bs,
    num_train_epochs=5,
    save_only_model = True,
    report_to = "none",
)

In [ ]:
user_prompt = traindata_sampled["prompt"][i]
assistant_response = traindata_sampled["response"][i]
formatted_messages = [
    {"role": "user", "content": user_prompt},
    {"role": "assistant", "content": assistant_response},
]

In [ ]:
tokenized_datasets = df.map(tokenize_function, batched=True, num_proc=4, remove_columns=["text"])
lm_datasets = tokenized_datasets.map(
group_texts,
batched=True,
batch_size=1024,
num_proc=4,
)

In [ ]:
trainer = Trainer( #fine-tuning, only LoRA parameters (A,B) are updated; C is frozen
    model=model,
    args=training_args,
    train_dataset=lm_datasets,
    )
trainer.train()